# Homework 1 – Agentic RAG

Building a RAG system from scratch over the course lesson pages, then making it agentic.

**Knowledge base:** All markdown lesson files from the `llm-zoomcamp` repo at commit `8c1834d`.  
**Model:** `gpt-5.4-mini` via OpenAI Responses API.  
**API key:** `OPENAI_KEY` loaded from `.env` at the project root. (create one if you want to test the solution)

## Setup

In [1]:
import json
import os

from dotenv import load_dotenv
from openai import OpenAI
from gitsource import GithubRepositoryDataReader, chunk_documents
from minsearch import Index

load_dotenv(dotenv_path="../.env")

client = OpenAI(api_key=os.environ["OPENAI_KEY"])
MODEL = "gpt-5.4-mini"

### Load lesson pages

Download all markdown files that live under a `/lessons/` folder from the course repo.
We pin to commit `8c1834d` so everyone works with the exact same data.

In [2]:
reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()

documents = [file.parse() for file in files]
print(f"Downloaded {len(documents)} documents")
print("Sample:", documents[0]["filename"])

Downloaded 72 documents
Sample: 01-agentic-rag/lessons/01-intro.md


In [9]:
from pprint import pprint

pprint(documents)

[{'content': '# Introduction\n'
             '\n'
             'Video: [Watch this '
             'lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n'
             '\n'
             "In this module, we'll build a working Retrieval-Augmented\n"
             'Generation (RAG) system from scratch, step by step.\n'
             '\n'
             'We write everything in plain Python. We build a small search '
             'index by\n'
             'hand and call the LLM ourselves. I want you to see every piece '
             'first.\n'
             'That way you know what a framework does for you before you reach '
             'for\n'
             'one.\n'
             '\n'
             'Places where you can find me:\n'
             '\n'
             '- [My substack](https://alexeyondata.substack.com/)\n'
             '- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n'
             '- [X](https://x.com/Al_Grigor)\n'
             '\n'
       

---
## Q1. How many lesson pages?

Count the number of documents fetched above.

In [3]:
print(f"Number of lesson pages: {len(documents)}")

Number of lesson pages: 72


---
## Q2. Indexing and searching

Index the documents with `minsearch`:
- `content` → text field
- `filename` → keyword field

Then search for: *"How does the agentic loop keep calling the model until it stops?"*

In [4]:
index = Index(
    text_fields=["content"],
    keyword_fields=["filename"],
)
index.fit(documents)

QUERY = "How does the agentic loop keep calling the model until it stops?"

results = index.search(QUERY, num_results=5)

print(f"First result: {results[0]['filename']}")
print("\nTop 5 results:")
for i, r in enumerate(results, 1):
    print(f"  {i}. {r['filename']}")

First result: 01-agentic-rag/lessons/14-agentic-loop.md

Top 5 results:
  1. 01-agentic-rag/lessons/14-agentic-loop.md
  2. 01-agentic-rag/lessons/15-frameworks.md
  3. 01-agentic-rag/lessons/13-function-calling.md
  4. 01-agentic-rag/lessons/11-agents-intro.md
  5. 01-agentic-rag/lessons/16-other-frameworks.md


---
## Q3. RAG

Build a RAG over the full-page index and answer the same query.  
We modify the LLM call to return the full response so we can read `usage.input_tokens`.

In [5]:
INSTRUCTIONS = """
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context, respond with "I don't know."
""".strip()

PROMPT_TEMPLATE = """
QUESTION: {question}

CONTEXT:
{context}
""".strip()


def build_context(search_results):
    lines = []
    for doc in search_results:
        lines.append(f"File: {doc['filename']}")
        lines.append(doc["content"])
        lines.append("")
    return "\n".join(lines).strip()


def rag(query, idx):
    search_results = idx.search(query, num_results=5)
    context = build_context(search_results)
    prompt = PROMPT_TEMPLATE.format(question=query, context=context)

    response = client.responses.create(
        model=MODEL,
        input=[
            {"role": "developer", "content": INSTRUCTIONS},
            {"role": "user", "content": prompt},
        ],
    )
    return response.output_text, response.usage


answer, usage = rag(QUERY, index)

print("Answer:")
print(answer)
print(f"\nInput tokens:  {usage.input_tokens}")
print(f"Output tokens: {usage.output_tokens}")

Answer:
The loop keeps calling the model with a `while True` loop and stops only when the model returns a response with **no function calls**.

In the lesson, that’s controlled by:

- setting `has_function_calls = False` at the start of each iteration
- checking each item in `response.output`
- if any item is a `function_call`, the code runs the tool and sets `has_function_calls = True`
- after processing the response, it breaks out of the loop only if `has_function_calls == False`

So the exit condition is: **no tool calls this turn means the agent is done**.

Input tokens:  7125
Output tokens: 131


---
## Q4. Chunking

Split each page into overlapping chunks with `size=2000` characters and `step=1000`.  
Consecutive chunks overlap by 1000 characters so no passage is split across a boundary.

In [6]:
chunks = chunk_documents(documents, size=2000, step=1000)

print(f"Documents: {len(documents)}")
print(f"Chunks:    {len(chunks)}")
print("\nSample chunk fields:", list(chunks[0].keys()))

Documents: 72
Chunks:    295

Sample chunk fields: ['start', 'content', 'filename']


---
## Q5. RAG with chunking

Index the chunks (same schema: `content` text, `filename` keyword) and run the same query.  
Compare `input_tokens` with Q3 to see how much smaller the prompt became.

In [7]:
chunk_index = Index(
    text_fields=["content"],
    keyword_fields=["filename"],
)
chunk_index.fit(chunks)

answer_chunked, usage_chunked = rag(QUERY, chunk_index)

print("Answer:")
print(answer_chunked)
print(f"\nInput tokens (full pages): {usage.input_tokens}")
print(f"Input tokens (chunks):     {usage_chunked.input_tokens}")
print(f"Reduction factor:          {usage.input_tokens / usage_chunked.input_tokens:.1f}x")

Answer:
The loop keeps calling the model in a `while True` loop and checks whether any `function_call` items were returned.

- If the model returns one or more function calls, the code runs them, adds the results to `messages`, and loops again.
- If the model returns only a final `message` and no function calls, `has_function_calls` stays `False`, and the loop `break`s.

So the stopping condition is: **no function calls in the current turn**.

Input tokens (full pages): 7125
Input tokens (chunks):     2308
Reduction factor:          3.1x


---
## Q6. Turning it into an agent

Give the LLM a `search` tool backed by the chunk index and let it decide when to search.  
We run the handwritten agentic loop (lesson 14) and count how many times `search` is called.

**Instructions:**
> You're a course teaching assistant. Answer the student's question using the search tool.
> Make multiple searches with different keywords before answering.

**Query:**
> How does the agentic loop work, and how is it different from plain RAG?

In [8]:
def search(query: str) -> list[dict]:
    """Search the course lessons for content matching the given query."""
    return chunk_index.search(query, num_results=5)


search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the course lessons for content matching the given query.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {"type": "string", "description": "The search query string."}
        },
        "required": ["query"],
        "additionalProperties": False,
    },
}


def make_call(call):
    args = json.loads(call.arguments)
    result = search(**args)
    return {
        "type": "function_call_output",
        "call_id": call.call_id,
        "output": json.dumps(result, indent=2),
    }


agent_instructions = (
    "You're a course teaching assistant. Answer the student's question using the "
    "search tool. Make multiple searches with different keywords before answering."
)

agent_query = "How does the agentic loop work, and how is it different from plain RAG?"

messages = [
    {"role": "developer", "content": agent_instructions},
    {"role": "user", "content": agent_query},
]

search_call_count = 0

for iteration in range(1, 20):
    print(f"--- iteration {iteration} ---")
    has_function_calls = False

    response = client.responses.create(
        model=MODEL,
        input=messages,
        tools=[search_tool],
    )
    messages.extend(response.output)

    for item in response.output:
        if item.type == "function_call":
            args = json.loads(item.arguments)
            print(f"  search({args['query']!r})")
            messages.append(make_call(item))
            has_function_calls = True
            search_call_count += 1
        elif item.type == "message":
            print("\nASSISTANT:")
            print(item.content[0].text)

    if not has_function_calls:
        break

print(f"\nTotal search calls: {search_call_count}")

--- iteration 1 ---
  search('agentic loop RAG lesson')
  search('agentic loop plain RAG')
  search('RAG agentic loop retrieval generation')
  search('agentic loop iterative retrieval tool use')
--- iteration 2 ---

ASSISTANT:
The **agentic loop** is the repeated cycle where the LLM is allowed to decide what to do next, instead of you hard-coding a single retrieve-then-answer path.

### How it works
From the course:
- You start with a **while loop**
- The LLM gets the user question
- If it returns a **tool/function call**, you execute that tool
- You send the tool result back to the LLM
- The LLM can then:
  - call another tool
  - refine its search terms
  - ask a clarifying question
  - or stop and give a final answer

So the loop looks like:

1. user asks a question  
2. LLM decides whether to use a tool  
3. if yes, run the tool  
4. feed the result back into the LLM  
5. repeat until the model stops calling tools

The key idea is that the **LLM controls the flow**.

### How it dif